# **Automobile Data Preprocessing Pipeline: Complete Guide**

## **1. Missing Data Handling**
- **Problem**: Dataset contains missing values marked as '?' in critical columns like price, horsepower, and normalized-losses
- **Solution**: Implemented multiple imputation strategies:
  - *Mean/Median Imputation*: For numerical columns like price and horsepower
  - *KNN Imputation*: Uses similar records to estimate missing values
  - *Mode Imputation*: For categorical variables like number-of-doors
- **Impact**: Prevents data loss while maintaining dataset integrity

## **2. Outlier Detection & Treatment**
- **Detection Methods**: 
  - *Z-score Analysis*: Identifies values beyond 3 standard deviations
  - *IQR Method*: Uses quartile ranges to detect extreme values
- **Treatment**: Capping outliers in key features like price, engine-size, and horsepower using IQR bounds
- **Benefit**: Prevents skewed model predictions and improves robustness

## **3. Categorical Data Encoding**
- **Label Encoding**: Applied to ordinal variables with inherent order:
  - 'num-of-cylinders' (two, four, six, eight)
  - 'num-of-doors' (two, four)
- **One-Hot Encoding**: Used for nominal variables without order:
  - 'make' (audi, bmw, toyota)
  - 'fuel-type' (gas, diesel)
  - 'body-style' (sedan, hatchback, convertible)
- **Result**: Converts text data to numerical format ML models can process

## **4. Data Scaling & Normalization**
- **Standardization (Z-score)**:
  - Transforms data to mean=0, standard deviation=1
  - Ideal for algorithms assuming normal distribution
- **Min-Max Normalization**:
  - Scales data to [0,1] range
  - Preserves original data relationships
- **Purpose**: Ensures all features contribute equally to model training

## **5. Feature Engineering & Selection**
- **New Feature Creation**:
  - *Power-to-weight ratio*: horsepower/curb-weight
  - *Size volume*: length × width × height
  - *Efficiency score*: average of city and highway MPG
- **Feature Selection Methods**:
  - *Correlation Analysis*: Keeps features highly correlated with price
  - *Recursive Feature Elimination*: Selects top 15 most important features
- **Advantage**: Improves model performance and reduces overfitting

## **6. Imbalanced Data Handling**
- **Problem**: Classification tasks often have uneven class distribution
- **SMOTE Oversampling**: Creates synthetic minority class samples
- **Random Undersampling**: Reduces majority class instances
- **Application**: Converts regression to classification by creating expensive/affordable categories
- **Benefit**: Balances class distribution for better classification accuracy

This comprehensive pipeline transforms raw automobile data into an optimized format ready for machine learning, addressing common data quality issues while enhancing predictive features for accurate price forecasting and vehicle classification.

In [10]:
!pip install imblearn


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('data/Automobile_data.csv')

print("Dataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (205, 26)

Missing Values:
symboling            0
normalized-losses    0
make                 0
fuel-type            0
aspiration           0
num-of-doors         0
body-style           0
drive-wheels         0
engine-location      0
wheel-base           0
length               0
width                0
height               0
curb-weight          0
engine-type          0
num-of-cylinders     0
engine-size          0
fuel-system          0
bore                 0
stroke               0
compression-ratio    0
horsepower           0
peak-rpm             0
city-mpg             0
highway-mpg          0
price                0
dtype: int64


**1. Handling Missing Data (Imputation Techniques)**

In [2]:
# Create a copy for imputation
df_imputed = df.copy()

# Identify numerical and categorical columns
numerical_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_imputed.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

# Handle '?' values - replace with NaN
df_imputed = df_imputed.replace('?', np.nan)

# Convert numerical columns to proper dtype
for col in ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'price']:
    df_imputed[col] = pd.to_numeric(df_imputed[col], errors='coerce')

# Update numerical columns after conversion
numerical_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_imputed.select_dtypes(include=['object']).columns.tolist()

print(f"\nMissing values after cleaning:")
print(df_imputed.isnull().sum().sort_values(ascending=False).head(10))

Numerical columns: ['symboling', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-size', 'compression-ratio', 'city-mpg', 'highway-mpg']
Categorical columns: ['normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 'drive-wheels', 'engine-location', 'engine-type', 'num-of-cylinders', 'fuel-system', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'price']

Missing values after cleaning:
normalized-losses    41
price                 4
stroke                4
bore                  4
peak-rpm              2
num-of-doors          2
horsepower            2
engine-type           0
highway-mpg           0
city-mpg              0
dtype: int64


**Apply Different Imputation Techniques**

In [3]:
# Strategy 1: Mean/Median Imputation for numerical, Mode for categorical
df_mean_imputed = df_imputed.copy()

# Numerical - mean imputation
num_imputer_mean = SimpleImputer(strategy='mean')
df_mean_imputed[numerical_cols] = num_imputer_mean.fit_transform(df_mean_imputed[numerical_cols])

# Categorical - mode imputation
cat_imputer_mode = SimpleImputer(strategy='most_frequent')
df_mean_imputed[categorical_cols] = cat_imputer_mode.fit_transform(df_mean_imputed[categorical_cols])

# Strategy 2: KNN Imputation for numerical columns
df_knn_imputed = df_imputed.copy()
knn_imputer = KNNImputer(n_neighbors=5)
df_knn_imputed[numerical_cols] = knn_imputer.fit_transform(df_knn_imputed[numerical_cols])
df_knn_imputed[categorical_cols] = cat_imputer_mode.fit_transform(df_knn_imputed[categorical_cols])

print("Missing values after imputation:")
print("Mean imputation:", df_mean_imputed.isnull().sum().sum())
print("KNN imputation:", df_knn_imputed.isnull().sum().sum())

Missing values after imputation:
Mean imputation: 0
KNN imputation: 0


**2. Outlier Detection and Handling**

In [4]:
# Use the mean imputed dataset for outlier handling
df_outlier_handled = df_mean_imputed.copy()

def detect_outliers_zscore(data, threshold=3):
    """Detect outliers using Z-score method"""
    z_scores = np.abs(stats.zscore(data.select_dtypes(include=[np.number])))
    return (z_scores > threshold).sum(axis=0)

def handle_outliers_iqr(data, column):
    """Handle outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap outliers
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    return data

print("Outliers detected (Z-score > 3):")
outlier_counts = detect_outliers_zscore(df_outlier_handled)
print(outlier_counts[outlier_counts > 0])

# Handle outliers in key numerical columns
outlier_columns = ['price', 'engine-size', 'horsepower', 'curb-weight']
for col in outlier_columns:
    if col in df_outlier_handled.columns:
        df_outlier_handled = handle_outliers_iqr(df_outlier_handled, col)

print("\nOutliers after handling:")
outlier_counts_after = detect_outliers_zscore(df_outlier_handled)
print(outlier_counts_after[outlier_counts_after > 0])

Outliers detected (Z-score > 3):
normalized-losses    2
wheel-base           1
engine-size          5
stroke               3
compression-ratio    9
horsepower           2
peak-rpm             2
city-mpg             3
highway-mpg          2
price                5
dtype: int64

Outliers after handling:
normalized-losses    2
wheel-base           1
stroke               3
compression-ratio    9
peak-rpm             2
city-mpg             3
highway-mpg          2
dtype: int64


**3. Encoding Categorical Data**

In [5]:
df_encoded = df_outlier_handled.copy()

# Label Encoding for ordinal-like categorical variables
label_encoders = {}
label_encode_cols = ['num-of-doors', 'num-of-cylinders']  # These have inherent order

for col in label_encode_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le

# One-Hot Encoding for nominal categorical variables
onehot_cols = ['make', 'fuel-type', 'aspiration', 'body-style', 'drive-wheels', 
               'engine-location', 'engine-type', 'fuel-system']

# Filter columns that exist in dataset
onehot_cols = [col for col in onehot_cols if col in df_encoded.columns]

df_encoded = pd.get_dummies(df_encoded, columns=onehot_cols, prefix=onehot_cols)

print(f"Shape after encoding: {df_encoded.shape}")
print(f"Number of columns increased from {df.shape[1]} to {df_encoded.shape[1]}")

Shape after encoding: (205, 69)
Number of columns increased from 26 to 69


**4. Data Scaling & Normalization**

In [6]:
# Separate features and target
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

# Identify numerical columns for scaling (excluding target)
numerical_features = X.select_dtypes(include=[np.number]).columns

# Standardization (Z-score Normalization)
scaler_standard = StandardScaler()
X_standardized = X.copy()
X_standardized[numerical_features] = scaler_standard.fit_transform(X_standardized[numerical_features])

# Min-Max Scaling
scaler_minmax = MinMaxScaler()
X_minmax = X.copy()
X_minmax[numerical_features] = scaler_minmax.fit_transform(X_minmax[numerical_features])

print("Original data range:")
print(f"Price: {X['symboling'].min():.2f} to {X['symboling'].max():.2f}")
print("\nAfter Standardization:")
print(f"Price: {X_standardized['symboling'].min():.2f} to {X_standardized['symboling'].max():.2f}")
print("\nAfter Min-Max Scaling:")
print(f"Price: {X_minmax['symboling'].min():.2f} to {X_minmax['symboling'].max():.2f}")

Original data range:
Price: -2.00 to 3.00

After Standardization:
Price: -2.28 to 1.74

After Min-Max Scaling:
Price: 0.00 to 1.00


**5. Feature Selection & Feature Engineering**

In [7]:
# Use standardized data for feature selection
X_selected = X_standardized.copy()

# Feature Engineering: Create new features
X_selected['power_to_weight'] = X_selected['horsepower'] / X_selected['curb-weight']
X_selected['size_ratio'] = X_selected['length'] * X_selected['width'] * X_selected['height']
X_selected['efficiency_score'] = (X_selected['city-mpg'] + X_selected['highway-mpg']) / 2

# Method 1: Correlation-based feature selection
correlation_with_target = X_selected.corrwith(y).abs().sort_values(ascending=False)
high_corr_features = correlation_with_target[correlation_with_target > 0.3].index.tolist()

print("Top correlated features with price:")
print(correlation_with_target.head(10))

# Method 2: Recursive Feature Elimination
rfe_selector = RFE(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42),
    n_features_to_select=15
)

# Fit RFE (using non-standardized numerical data for tree-based models)
X_for_rfe = X.copy()
X_for_rfe['power_to_weight'] = X_for_rfe['horsepower'] / X_for_rfe['curb-weight']
X_for_rfe['size_ratio'] = X_for_rfe['length'] * X_for_rfe['width'] * X_for_rfe['height']

rfe_selector.fit(X_for_rfe.select_dtypes(include=[np.number]), y)
rfe_features = X_for_rfe.select_dtypes(include=[np.number]).columns[rfe_selector.support_].tolist()

print(f"\nRFE selected {len(rfe_features)} features:")
print(rfe_features)

# Select final features
final_features = list(set(high_corr_features[:10] + rfe_features))
X_final = X_selected[final_features]

print(f"\nFinal dataset shape: {X_final.shape}")

Top correlated features with price:
curb-weight         0.850489
engine-size         0.840976
horsepower          0.796230
width               0.753471
highway-mpg         0.715295
length              0.713706
efficiency_score    0.710651
city-mpg            0.695785
drive-wheels_rwd    0.668013
drive-wheels_fwd    0.625107
dtype: float64

RFE selected 15 features:
['normalized-losses', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-size', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'power_to_weight']

Final dataset shape: (205, 18)


**6. Handling Imbalanced Data (for Classification)**

In [11]:
# Create a classification problem: Expensive vs Affordable cars
median_price = y.median()
y_class = (y > median_price).astype(int)  # 1 for expensive, 0 for affordable

print("Class distribution:")
print(y_class.value_counts())
print(f"Imbalance ratio: {y_class.mean():.3f}")

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Prepare data for sampling (use original features before encoding for demonstration)
X_sampling = df_outlier_handled.select_dtypes(include=[np.number]).drop('price', axis=1)

# SMOTE (Synthetic Minority Oversampling Technique)
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_sampling, y_class)

# Undersampling
undersampler = RandomUnderSampler(random_state=42)
X_under, y_under = undersampler.fit_resample(X_sampling, y_class)

print("\nBefore sampling:", Counter(y_class))
print("After SMOTE:", Counter(y_smote))
print("After Undersampling:", Counter(y_under))

Class distribution:
price
0    103
1    102
Name: count, dtype: int64
Imbalance ratio: 0.498

Before sampling: Counter({0: 103, 1: 102})
After SMOTE: Counter({1: 103, 0: 103})
After Undersampling: Counter({0: 102, 1: 102})


**Complete Preprocessing Pipeline**

In [9]:
def automobile_preprocessing_pipeline(df, target_col='price', sampling_method=None):
    """
    Complete preprocessing pipeline for automobile data
    """
    df_processed = df.copy()
    
    # Step 1: Handle missing values
    df_processed = df_processed.replace('?', np.nan)
    
    # Convert numerical columns
    for col in ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'price']:
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
    
    # Impute numerical with mean, categorical with mode
    numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
    categorical_cols = df_processed.select_dtypes(include=['object']).columns
    
    num_imputer = SimpleImputer(strategy='mean')
    cat_imputer = SimpleImputer(strategy='most_frequent')
    
    df_processed[numerical_cols] = num_imputer.fit_transform(df_processed[numerical_cols])
    df_processed[categorical_cols] = cat_imputer.fit_transform(df_processed[categorical_cols])
    
    # Step 2: Feature engineering
    df_processed['power_to_weight'] = df_processed['horsepower'] / df_processed['curb-weight']
    df_processed['size_volume'] = df_processed['length'] * df_processed['width'] * df_processed['height']
    
    # Step 3: Encoding
    # Label encoding for ordinal features
    if 'num-of-cylinders' in df_processed.columns:
        cylinder_map = {'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'eight': 8, 'twelve': 12}
        df_processed['num-of-cylinders'] = df_processed['num-of-cylinders'].map(cylinder_map)
    
    # One-hot encoding for nominal features
    categorical_to_encode = ['make', 'fuel-type', 'aspiration', 'body-style', 
                           'drive-wheels', 'engine-location', 'engine-type', 'fuel-system']
    categorical_to_encode = [col for col in categorical_to_encode if col in df_processed.columns]
    
    df_processed = pd.get_dummies(df_processed, columns=categorical_to_encode, prefix=categorical_to_encode)
    
    # Step 4: Prepare features and target
    X = df_processed.drop(target_col, axis=1)
    y = df_processed[target_col]
    
    # Step 5: Scaling
    numerical_features = X.select_dtypes(include=[np.number]).columns
    scaler = StandardScaler()
    X[numerical_features] = scaler.fit_transform(X[numerical_features])
    
    return X, y, scaler

# Apply complete pipeline
X_processed, y_processed, fitted_scaler = automobile_preprocessing_pipeline(df)

print("Final processed dataset:")
print(f"Features shape: {X_processed.shape}")
print(f"Target shape: {y_processed.shape}")
print(f"Missing values: {X_processed.isnull().sum().sum()}")

Final processed dataset:
Features shape: (205, 70)
Target shape: (205,)
Missing values: 0
